In [9]:
import pandas as pd
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import tensorflow as tf
import matplotlib as plt
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

In [5]:
# gloabal parameters
IMG_SIZE = 224          # ResNet/EfficientNet typically use 224x224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE  # Let TensorFlow decide parallelism

In [2]:
# load training and validation datasets
train_ds = tf.data.Dataset.load(r"train_and_val_datasets\train_ds")
val_ds = tf.data.Dataset.load(r"train_and_val_datasets\val_ds")

In [6]:
# Load the base model without the top classification layer
base_model = EfficientNetB0(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,       # Remove the original 1000-class head
    weights='imagenet'       # Use weights pre-trained on ImageNet
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [7]:
# Freeze the base model's layers as to not destroy the learned weights
base_model.trainable = False

In [8]:
# Build the new model
model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    
    # Base model (frozen)
    base_model,

    # Global average pooling: converts feature maps to a single vector per image
    layers.GlobalAveragePooling2D(),

    # A small dropout for regularisation
    layers.Dropout(0.3),

    # The final classification layer with 7 units (one per class)
    layers.Dense(7, activation='softmax')
])


**Preprocessing note:**  
EfficientNet’s official preprocessing expects pixel values to be in the `[0, 255]` range and then applies a specific scaling (like `tf.keras.applications.efficientnet.preprocess_input`). To keep things simple, we will add that as a Lambda layer inside the model.:

In [10]:
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
# Our images are in [0,1] after loading; EfficientNet expects [0,255] then scaling.
# So we first multiply by 255, then apply the official preprocess function.
x = layers.Lambda(lambda img: effnet_preprocess(img * 255.0))(inputs)
x = base_model(x, training=False)  # ensure base runs in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(7, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

-   `base_model.trainable = False` freezes all layers so they won’t be updated during the first training phase. This prevents wrecking the pre‑trained features while our new top layers learn.
-   `GlobalAveragePooling2D` reduces the spatial dimensions (7×7×1280 for EfficientNetB0) down to a single vector of length 1280 by averaging each feature map.
-   `Dropout` randomly drops 30% of the connections during training, making the model less likely to overfit.
-   The final `Dense` layer with `softmax` outputs a probability distribution over the 7 classes.

In [11]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

using tensorflow-directml because recieived this Warning:
```
WARNING:tensorflow:TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.
```

In [ ]:
# First, uninstall tensorflow if installed
pip uninstall tensorflowK

# Then install tensorflow-directml (includes TF 2.10 + DirectML plugin)
pip install tensorflow-directml